In [1]:
import sys, platform #确认运行环境与依赖
print("Python:", sys.version)
print("Platform:", platform.platform())

import pandas as pd
import numpy as np
import openpyxl
from rapidfuzz import fuzz, process

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("openpyxl:", openpyxl.__version__)

Python: 3.11.14 | packaged by conda-forge | (main, Jan 27 2026, 00:01:01) [Clang 19.1.7 ]
Platform: macOS-26.3-arm64-arm-64bit
pandas: 3.0.1
numpy: 2.4.2
openpyxl: 3.1.5


In [2]:
from pathlib import Path
#定位目录 & 自动发现 Excel
base_dir = Path("/Users/xuyali/Documents/个人资料/Work/自媒体/Total Asia")
base_dir.exists(), str(base_dir)
# 列出目录下的 excel 文件，确认 A/B 具体文件名
excel_files = sorted(list(base_dir.glob("*.xlsx")) + list(base_dir.glob("*.xls")))
[(f.name, f.stat().st_size) for f in excel_files]

[('ERP hh001.xls', 53760), ('hh001.xlsx', 33533), ('~$hh001.xlsx', 165)]

In [8]:
# === 你在这里指定文件名（从上一步列表中复制）===
a_file = base_dir / "hh001.xlsx"
b_file = base_dir / "ERP hh001.xls"

df_a = pd.read_excel(a_file)
df_b = pd.read_excel(b_file)

df_a.columns = [c.strip() for c in df_a.columns]
df_b.columns = [c.strip() for c in df_b.columns]

print("A shape:", df_a.shape)
print("B shape:", df_b.shape)
df_a.head(3), df_b.head(3)

A shape: (101, 11)
B shape: (101, 10)


(      CODE   QTY  PRICE  PACK SIZE                        DESCRIPTION  \
 0  12991.0  80.0   8.19       2x5L          SARSONS Malt Vinegar 2x5L   
 1  51061.0  50.0  20.99   20kg bag        PHOENIX USA Long Grain Rice   
 2  71711.0  50.0  18.30  7.7kg box  LUCKY BOAT No.2 Extra Fine Noodle   
 
    Unnamed: 5 Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9    V  
 0         NaN  沙臣士牌桶裝大麥醋         NaN         NaN         NaN  1.0  
 1         NaN   雙鳳牌美國絲苗米         NaN         NaN         NaN  1.0  
 2         NaN   帆船牌免蒸炒底麵         NaN         NaN         NaN  1.0  ,
    Unnamed: 0  Product  Suppl category  \
 0           1    401.0         44581.0   
 1           2   2221.0         18221.0   
 2           3   2933.0         75471.0   
 
                                          Description   Size  Pack Store  \
 0                 Bullhead Barbecue BBQ Sauce 牛头牌沙茶酱   737g  12.0   0/0   
 1         MAGGI Thick Soy Sauce Nuoc Tuong Dau美極越南醬油  700ML  12.0   0/6   
 2  Yummy House Sliced 

In [9]:
# 统一字段名（把 B 的 Suppl category/Suppl 统一成 suppl_category）
rename_a = {
    "CODE": "code",
    "QTY": "qty_a",
    "PRICE": "price_a",
    "PACK SIZE": "pack_size_a",
    "DESCRIPTION": "desc_a",
}

# B: 兼容不同列名
rename_b = {
    "Product": "product",
    "Description": "desc_b",
    "Size": "size_b",
    "Pack": "pack_b",
    "Store": "store",
    "Price": "price_b",
    "Qty.": "qty_b",
    "Qty": "qty_b",
    "Suppl": "suppl_category",
    "Suppl category": "suppl_category",
    "Suppl Category": "suppl_category",
}

df_a = df_a.rename(columns=rename_a)
df_b = df_b.rename(columns=rename_b)

required_a = ["code", "desc_a", "pack_size_a", "qty_a", "price_a"]
required_b = ["product", "suppl_category", "desc_b", "size_b"]

print("A missing required cols:", [c for c in required_a if c not in df_a.columns])
print("B missing required cols:", [c for c in required_b if c not in df_b.columns])

A missing required cols: []
B missing required cols: []


In [10]:
import re

def to_int_safe(x):
    return pd.to_numeric(x, errors="coerce").astype("Int64")

def to_money(x):
    if isinstance(x, str):
        x = x.replace("£", "").strip()
    return pd.to_numeric(x, errors="coerce")

# 编号字段：抽取数字（避免 "44581 " 或 "ID:44581"）
def extract_digits(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    m = re.search(r"\d+", s)
    return int(m.group()) if m else np.nan

df_a["code"] = df_a["code"].apply(extract_digits).astype("Int64")
df_b["suppl_category"] = df_b["suppl_category"].apply(extract_digits).astype("Int64")

df_a["qty_a"] = to_int_safe(df_a["qty_a"])
df_a["price_a"] = df_a["price_a"].apply(to_money)

# 基础缺失统计
print(df_a[["code","qty_a","price_a","desc_a","pack_size_a"]].isna().mean().sort_values(ascending=False))
print(df_b[["product","suppl_category","desc_b","size_b"]].isna().mean().sort_values(ascending=False))

qty_a          0.306931
price_a        0.306931
code           0.019802
pack_size_a    0.019802
desc_a         0.000000
dtype: float64
product           0.009901
suppl_category    0.009901
desc_b            0.009901
size_b            0.009901
dtype: float64


In [11]:
def english_only(text: str) -> str:
    if pd.isna(text):
        return ""
    s = str(text).lower()
    # 把各种撇号统一掉，然后去掉非字母数字空格
    s = s.replace("’", "'")
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokens(text: str):
    s = english_only(text)
    if not s:
        return []
    return [t for t in s.split(" ") if t]

df_a["desc_a_en"] = df_a["desc_a"].apply(english_only)
df_b["desc_b_en"] = df_b["desc_b"].apply(english_only)

df_a["tok_a"] = df_a["desc_a_en"].apply(tokens)
df_b["tok_b"] = df_b["desc_b_en"].apply(tokens)

# 快速人工检查：随机抽样
df_a[["desc_a","desc_a_en","tok_a"]].head(10)

,desc_a,desc_a_en,tok_a
0,SARSONS Malt Vinegar 2x5L,sarsons malt vinegar 2x5l,"[sarsons, malt, vinegar, 2x5l]"
1,PHOENIX USA Long Grain Rice,phoenix usa long grain rice,"[phoenix, usa, long, grain, rice]"
2,LUCKY BOAT No.2 Extra Fine Noodle,lucky boat no 2 extra fine noodle,"[lucky, boat, no, 2, extra, fine, noodle]"
3,TIGER Beer Bottles,tiger beer bottles,"[tiger, beer, bottles]"
4,807M03Y SAU TAO Japanese Ramen Noodles,807m03y sau tao japanese ramen noodles,"[807m03y, sau, tao, japanese, ramen, noodles]"
5,DOUBLE HAPPINESS Washing Up Liquid,double happiness washing up liquid,"[double, happiness, washing, up, liquid]"
6,DOUBLE HAPPINESS Bleach,double happiness bleach,"[double, happiness, bleach]"
7,YELLOW KIRIN Wheat Flour 50lbs,yellow kirin wheat flour 50lbs,"[yellow, kirin, wheat, flour, 50lbs]"
8,EVIAN Mineral Water 24X500ML,evian mineral water 24x500ml,"[evian, mineral, water, 24x500ml]"
9,HONG KONG Double Dark Soy Sauce (50lbs),hong kong double dark soy sauce 50lbs,"[hong, kong, double, dark, soy, sauce, 50lbs]"


In [12]:
# 提取 "数字+单位" 组合（支持小数）
UNIT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(kg|g|ml|l|pcs)\b", re.IGNORECASE)

def parse_pack_size(pack: str):
    if pd.isna(pack):
        return ""
    s = str(pack).strip().lower()
    s = s.replace("×", "x")  # 防止乘号
    # 若包含 x：取最后一段（x 后面的部分）再提取数字+单位
    if "x" in s:
        tail = s.split("x")[-1]
        m = UNIT_RE.search(tail)
        if m:
            num, unit = m.group(1), m.group(2)
            return f"{num}{unit}"
    # 否则：取第一个数字+单位
    m = UNIT_RE.search(s)
    if m:
        num, unit = m.group(1), m.group(2)
        return f"{num}{unit}"
    return ""

def norm_size(s):
    if pd.isna(s):
        return ""
    x = str(s).strip().lower().replace(" ", "")
    x = x.replace("litre", "l").replace("liter", "l")
    return x

df_a["pack_parsed"] = df_a["pack_size_a"].apply(parse_pack_size)
df_b["size_norm"] = df_b["size_b"].apply(norm_size)
df_a["pack_norm"] = df_a["pack_parsed"].apply(norm_size)

df_a[["pack_size_a","pack_parsed","pack_norm"]].head(20)

,pack_size_a,pack_parsed,pack_norm
0,2x5L,5l,5l
1,20kg bag,20kg,20kg
2,7.7kg box,7.7kg,7.7kg
3,24x330ml,330ml,330ml
4,12x540g bag,540g,540g
5,2x5L,5l,5l
6,2x5L,5l,5l
7,50lbs22.68kg,22.68kg,22.68kg
8,24x500ml,500ml,500ml
9,22.68kg drum,22.68kg,22.68kg


In [17]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz
import re

# ===== Checkpoint 6.0：输入表必须非空，关键列必须存在 =====
print("df_a shape:", df_a.shape, "df_b shape:", df_b.shape)
if df_a.shape[0] == 0:
    raise ValueError("df_a is empty. 请回到 Step 2/3 检查读取、重命名、过滤是否把数据清空了。")
if df_b.shape[0] == 0:
    raise ValueError("df_b is empty. 请回到 Step 2/3 检查读取、重命名、过滤是否把数据清空了。")

must_a = ["code","desc_a","desc_a_en","tok_a","pack_size_a","pack_norm","qty_a","price_a"]
must_b = ["product","suppl_category","desc_b","desc_b_en","tok_b","size_b","size_norm"]
missing_a = [c for c in must_a if c not in df_a.columns]
missing_b = [c for c in must_b if c not in df_b.columns]
if missing_a or missing_b:
    raise ValueError(f"Missing columns. A missing={missing_a}, B missing={missing_b}")

# ===== 工具函数 =====
def common_token_count(a_tokens, b_tokens):
    return len(set(a_tokens) & set(b_tokens))

def desc_similarity(a_text, b_text):
    return fuzz.token_set_ratio(a_text, b_text)

def norm_size(s):
    if pd.isna(s):
        return ""
    x = str(s).strip().lower().replace(" ", "")
    x = x.replace("litre", "l").replace("liter", "l")
    return x

# ===== 给 B 建编号索引 =====
b_by_code = df_b.dropna(subset=["suppl_category"]).groupby("suppl_category")

# ===== 主循环 =====
results = []
errors = []

for i, a in df_a.iterrows():
    try:
        a_code = a["code"]
        a_tok = a["tok_a"]
        a_text = a["desc_a_en"]
        a_pack = a["pack_norm"]

        # --- 候选1：编号候选（始终为 DataFrame） ---
        if pd.notna(a_code) and a_code in b_by_code.groups:
            code_candidates = df_b.loc[b_by_code.groups[a_code]].copy()
        else:
            code_candidates = df_b.iloc[0:0].copy()

        # --- 候选2：描述候选（共同词>=3；先保证正确性，后续可优化加速） ---
        commons = df_b["tok_b"].apply(lambda bt: common_token_count(a_tok, bt))
        desc_cands = df_b.loc[commons >= 3].copy()
        if not desc_cands.empty:
            desc_cands["common_words"] = commons[commons >= 3].values
            desc_cands["sim"] = desc_cands["desc_b_en"].apply(lambda bt: desc_similarity(a_text, bt))
            desc_cands = desc_cands.sort_values(["common_words", "sim"], ascending=[False, False])

        best_desc = None if desc_cands.empty else desc_cands.iloc[0]

        # --- 判定 ---
        match_code = False
        match_desc = best_desc is not None
        match_size = False
        rule_used = ""
        chosen = None

        # 先尝试 code + desc（必须共同词>=3）
        if (not code_candidates.empty) and (best_desc is not None):
            cc = code_candidates.copy()
            cc["common_words"] = cc["tok_b"].apply(lambda bt: common_token_count(a_tok, bt))
            cc = cc.loc[cc["common_words"] >= 3].copy()
            if not cc.empty:
                cc["sim"] = cc["desc_b_en"].apply(lambda bt: desc_similarity(a_text, bt))
                cc = cc.sort_values(["common_words", "sim"], ascending=[False, False])
                chosen = cc.iloc[0]
                match_code = True
                match_desc = True
                match_size = (a_pack != "" and a_pack == norm_size(chosen["size_b"]))
                rule_used = "code+desc"
            else:
                match_code = True  # 有编号候选但描述没过>=3门槛

        # 再尝试 desc + size
        if chosen is None and best_desc is not None:
            chosen = best_desc
            match_size = (a_pack != "" and a_pack == norm_size(chosen["size_b"]))
            if match_size:
                rule_used = "desc+size"

        confirmed = rule_used in ["code+desc", "desc+size"]

        results.append({
            "a_row": i,
            "A_CODE": a_code,
            "A_DESC": a["desc_a"],
            "A_PACK_SIZE": a["pack_size_a"],
            "A_PACK_PARSED": a_pack,
            "A_QTY": a["qty_a"],
            "A_PRICE": a["price_a"],

            "confirmed": confirmed,
            "rule_used": rule_used,
            "match_code": match_code,
            "match_desc": match_desc,
            "match_size": match_size,

            "B_Product": None if chosen is None else chosen["product"],
            "B_Suppl_category": None if chosen is None else chosen["suppl_category"],
            "B_Desc": None if chosen is None else chosen["desc_b"],
            "B_Size": None if chosen is None else chosen["size_b"],

            "common_words": None if chosen is None else common_token_count(a_tok, chosen["tok_b"]),
            "sim": None if chosen is None else desc_similarity(a_text, chosen["desc_b_en"]),
        })

    except Exception as e:
        errors.append({"a_row": i, "A_CODE": a.get("code", None), "error": repr(e)})
        # 同时也把这行作为未确认记录保留下来，避免结果集为空
        results.append({
            "a_row": i,
            "A_CODE": a.get("code", None),
            "A_DESC": a.get("desc_a", None),
            "A_PACK_SIZE": a.get("pack_size_a", None),
            "A_PACK_PARSED": a.get("pack_norm", None),
            "A_QTY": a.get("qty_a", None),
            "A_PRICE": a.get("price_a", None),
            "confirmed": False,
            "rule_used": "error",
            "match_code": False,
            "match_desc": False,
            "match_size": False,
            "B_Product": None,
            "B_Suppl_category": None,
            "B_Desc": None,
            "B_Size": None,
            "common_words": None,
            "sim": None,
        })

res = pd.DataFrame(results)

print("res shape:", res.shape)
print("res columns:", res.columns.tolist())
print("errors:", len(errors))

matched = res[res["confirmed"]].copy()
unmatched = res[~res["confirmed"]].copy()

print("Matched:", len(matched), "Unmatched:", len(unmatched), "Total:", len(res))

# 可选：查看错误明细
err_df = pd.DataFrame(errors)
err_df.head(10)

df_a shape: (101, 15) df_b shape: (101, 13)
res shape: (101, 18)
res columns: ['a_row', 'A_CODE', 'A_DESC', 'A_PACK_SIZE', 'A_PACK_PARSED', 'A_QTY', 'A_PRICE', 'confirmed', 'rule_used', 'match_code', 'match_desc', 'match_size', 'B_Product', 'B_Suppl_category', 'B_Desc', 'B_Size', 'common_words', 'sim']
errors: 0
Matched: 91 Unmatched: 10 Total: 101


""


In [18]:
# 看匹配率
print("Matched:", len(matched), "Unmatched:", len(unmatched), "Total:", len(res))
print("Matched %:", round(len(matched)/len(res)*100, 2))

# 人工抽查 20 条匹配
matched.sample(min(20, len(matched)))[
    ["A_CODE","A_DESC","B_Product","B_Desc","A_PACK_PARSED","B_Size","rule_used","common_words","sim"]
]

Matched: 91 Unmatched: 10 Total: 101
Matched %: 90.1


,A_CODE,A_DESC,B_Product,B_Desc,A_PACK_PARSED,B_Size,rule_used,common_words,sim
84,39111,JAWIRAT Fermented Fish Sauce Traditional,500930.0,JAWIRAT Fermented Fish Sauce 泰式发酵鱼露,350ml,350ml,code+desc,4.0,100.000000
81,60391,WANG Stick Rice Cake Tteokbokki Tteock,6836.0,34514WANG Stick Rice Cake Tteokboki Tteock王牌韩国年糕条,600g,600g,code+desc,4.0,80.000000
5,13591,DOUBLE HAPPINESS Washing Up Liquid,899515.0,DOUBLE HAPPINESS Washing Up Liquid-5L双喜牌甘油,5l,5L,code+desc,5.0,100.000000
28,37341,COCK BRAND Sweet Chilli Sauce for Chicken,321011.0,COCK Brand Sweet Chilli Sauce for Chicken 雄鸡牌甜辣鸡酱,350g,350g,code+desc,7.0,100.000000
20,66201,SIGNWIN Matcha Pudding Powder,1328.0,Singwin Matcha Pudding Powder抹茶味布丁粉,100g,100g,code+desc,3.0,96.551724
2,71711,LUCKY BOAT No.2 Extra Fine Noodle,997025.0,No2 Lucky Boat Noodle NO2 帆船炒底面,7.7kg,7.7kg,code+desc,3.0,89.473684
18,44581,BULLHEAD Barbecue Sauce,401.0,Bullhead Barbecue BBQ Sauce 牛头牌沙茶酱,737g,737g,code+desc,3.0,100.000000
76,53061,DL WW Sandwich Biscuit - Vanilla Flv,990116.0,WW Wantwant Sandwich Biscuit Vanilla 旺旺原味黑白配,56g,60g,code+desc,4.0,88.524590
47,67531,WANG Pickled Radish ( Cut ),968083.0,Wang Pickled Radish (Sliced) 黃蘿蔔大根 (切片),1kg,1KG,code+desc,3.0,90.476190
37,36161,PANTAI Ground Preserved Fish Sauce,791801.0,Pantai Ground Preserved Fish sauce,730ml,730ml,code+desc,5.0,100.000000


In [23]:
# 导出C 表：以 B.Product 为主键输出，并带入 A 的 qty/price
C = matched[[
    "B_Product","B_Suppl_category","B_Desc","B_Size",
    "A_CODE","A_DESC","A_PACK_SIZE","A_PACK_PARSED",
    "A_QTY","A_PRICE",
    "rule_used","common_words","sim"
]].rename(columns={
    "B_Product": "Product",
    "B_Suppl_category": "Suppl category",
    "B_Desc": "Description",
    "B_Size": "Size",
})

# 未匹配清单
U = unmatched[[
    "A_CODE","A_DESC","A_PACK_SIZE","A_PACK_PARSED","A_QTY","A_PRICE",
    "match_code","match_desc","match_size","common_words","sim"
]].copy()

out_dir = base_dir / "output"
out_dir.mkdir(exist_ok=True)

c_path = out_dir / "C_matched.xlsx"
u_path = out_dir / "C_unmatched.xlsx"

with pd.ExcelWriter(c_path, engine="openpyxl") as w:
    C.to_excel(w, index=False, sheet_name="C")

with pd.ExcelWriter(u_path, engine="openpyxl") as w:
    U.to_excel(w, index=False, sheet_name="Unmatched")

str(c_path), str(u_path)

('/Users/xuyali/Documents/个人资料/Work/自媒体/Total Asia/output/C_matched.xlsx',
 '/Users/xuyali/Documents/个人资料/Work/自媒体/Total Asia/output/C_unmatched.xlsx')

In [20]:
# ===== Step 7：生成 C（以 B 为底表，保持 B 顺序）=====

# 1) 从 matched 中取出 B_Product -> A_QTY/A_PRICE 映射
#    若重复匹配到同一 B_Product：保留第一条（可按需改策略）
map_df = matched[["B_Product", "A_QTY", "A_PRICE"]].copy()
map_df = map_df.dropna(subset=["B_Product"])
map_df = map_df.drop_duplicates(subset=["B_Product"], keep="first")

qty_map = dict(zip(map_df["B_Product"], map_df["A_QTY"]))
price_map = dict(zip(map_df["B_Product"], map_df["A_PRICE"]))

# 2) 以 B 为底表复制一份（保持原顺序）
C = df_b.copy()

# 3) 新增两列：QTY / PRICE（列名按你要求）
C["QTY"] = C["product"].map(qty_map)
C["PRICE"] = C["product"].map(price_map)

# 4) 输出未匹配清单（A 侧未确认匹配的行）
U = unmatched[[
    "A_CODE","A_DESC","A_PACK_SIZE","A_PACK_PARSED","A_QTY","A_PRICE",
    "match_code","match_desc","match_size","common_words","sim"
]].copy()

# 5) 导出
from pathlib import Path
out_dir = base_dir / "output"
out_dir.mkdir(exist_ok=True)

c_path = out_dir / "C_based_on_B.xlsx"
u_path = out_dir / "C_unmatched.xlsx"

with pd.ExcelWriter(c_path, engine="openpyxl") as w:
    C.to_excel(w, index=False, sheet_name="C")

with pd.ExcelWriter(u_path, engine="openpyxl") as w:
    U.to_excel(w, index=False, sheet_name="Unmatched")

print("Saved:", c_path)
print("Saved:", u_path)

# ===== Checkpoint：确认顺序与回填情况 =====
print("C rows == B rows:", len(C) == len(df_b))
print("Filled QTY count:", C["QTY"].notna().sum())
print("Filled PRICE count:", C["PRICE"].notna().sum())

Saved: /Users/xuyali/Documents/个人资料/Work/自媒体/Total Asia/output/C_based_on_B.xlsx
Saved: /Users/xuyali/Documents/个人资料/Work/自媒体/Total Asia/output/C_unmatched.xlsx
C rows == B rows: True
Filled QTY count: 64
Filled PRICE count: 64


In [21]:
# ===== Secondary pass: only for unmatched, require 1+2+3 all true =====

from rapidfuzz import fuzz

def common_token_count(a_tokens, b_tokens):
    return len(set(a_tokens) & set(b_tokens))

def desc_similarity(a_text, b_text):
    return fuzz.token_set_ratio(a_text, b_text)

def norm_size(s):
    if pd.isna(s):
        return ""
    x = str(s).strip().lower().replace(" ", "")
    x = x.replace("litre", "l").replace("liter", "l")
    return x

# B 按 suppl_category 建索引
b_by_code = df_b.dropna(subset=["suppl_category"]).groupby("suppl_category")

results2 = []

for _, r in unmatched.iterrows():
    a_code = r["A_CODE"]
    a_text = english_only(r["A_DESC"])
    a_tok  = tokens(r["A_DESC"])
    a_pack = norm_size(r["A_PACK_PARSED"])

    # 必须满足 1：code 在 B 里能找到候选
    if pd.isna(a_code) or (a_code not in b_by_code.groups):
        continue

    code_candidates = df_b.loc[b_by_code.groups[a_code]].copy()

    # 必须满足 2：共同词 >= 2（只在编号候选内做，避免全表误配）
    code_candidates["common_words"] = code_candidates["tok_b"].apply(lambda bt: common_token_count(a_tok, bt))
    cc = code_candidates.loc[code_candidates["common_words"] >= 2].copy()
    if cc.empty:
        continue

    # 用相似度辅助选最优
    cc["sim"] = cc["desc_b_en"].apply(lambda bt: desc_similarity(a_text, bt))
    cc = cc.sort_values(["common_words", "sim"], ascending=[False, False])
    chosen = cc.iloc[0]

    # 必须满足 3：size 匹配
    size_ok = (a_pack != "" and a_pack == norm_size(chosen["size_b"]))
    if not size_ok:
        continue

    # 三条都满足 -> 确认补匹配成功
    results2.append({
        **r.to_dict(),
        "confirmed": True,
        "rule_used": "code+desc(2w)+size",
        "match_code": True,
        "match_desc": True,
        "match_size": True,
        "B_Product": chosen["product"],
        "B_Suppl_category": chosen["suppl_category"],
        "B_Desc": chosen["desc_b"],
        "B_Size": chosen["size_b"],
        "common_words": int(chosen["common_words"]),
        "sim": float(chosen["sim"]),
    })

matched2 = pd.DataFrame(results2)

# 从 unmatched 里剔除已补匹配的 a_row（或用 A_CODE+A_DESC 也行）
if not matched2.empty:
    hit_rows = set(matched2["a_row"].tolist())
    unmatched2 = unmatched[~unmatched["a_row"].isin(hit_rows)].copy()
else:
    unmatched2 = unmatched.copy()

print("Secondary matched:", len(matched2))
print("Still unmatched:", len(unmatched2))


Secondary matched: 8
Still unmatched: 2


In [22]:
matched_all = pd.concat([matched, matched2], ignore_index=True)
unmatched_all = unmatched2

print("Total matched:", len(matched_all))
print("Total unmatched:", len(unmatched_all))

Total matched: 99
Total unmatched: 2


In [ ]:
matched_all = pd.concat([matched, matched2], ignore_index=True)
unmatched_all = unmatched2

print("Total matched:", len(matched_all))
print("Total unmatched:", len(unmatched_all))

In [24]:
# ===== Final C generation (B-order, only requested columns) =====

# 你前面如果已运行：
# matched_all = pd.concat([matched, matched2], ignore_index=True)
# unmatched_all = unmatched2
# 那么这里直接用 matched_all
use_matched = matched_all if "matched_all" in globals() else matched

# 1) 构造 Product -> A_QTY/A_PRICE 映射
map_df = use_matched[["B_Product", "A_QTY", "A_PRICE"]].copy()
map_df = map_df.dropna(subset=["B_Product"])

# 如果同一个 Product 被多条 A 记录命中：保留第一条（需要别的策略再说）
map_df = map_df.drop_duplicates(subset=["B_Product"], keep="first")

qty_map = dict(zip(map_df["B_Product"], map_df["A_QTY"]))
price_map = dict(zip(map_df["B_Product"], map_df["A_PRICE"]))

# 2) 以 B 为底表复制（保持 B 顺序）
C = df_b.copy()

# 3) 用 A 的数据覆盖/填充 Price 和 Qty（列名按你要求：Price / Qty）
#    注意：df_b 里可能叫 qty_b/price_b 或 Qty./Price；但我们最终输出就叫 Price/Qty
C["Qty"] = C["product"].map(qty_map)
C["Price"] = C["product"].map(price_map)

# 4) 只保留你指定的 8 列，并把列名调整成你想要的显示名
#    df_b 标准化后列名是：product, suppl_category, desc_b, size_b, pack_b, store
C_out = C.rename(columns={
    "product": "Product",
    "suppl_category": "Suppl category",
    "desc_b": "Description",
    "size_b": "Size",
    "pack_b": "Pack",
    "store": "Store",
})[["Product","Suppl category","Description","Size","Pack","Store","Price","Qty"]].copy()

# ===== Checkpoint: 是否都 match 上 =====
missing_qty = C_out["Qty"].isna().sum()
missing_price = C_out["Price"].isna().sum()
print("Rows:", len(C_out))
print("Missing Qty:", missing_qty)
print("Missing Price:", missing_price)

# 若你期望全部匹配，这里给出未回填的样本（按 Product）
if missing_qty > 0 or missing_price > 0:
    print("Sample rows not filled (first 20):")
    display(C_out[C_out["Qty"].isna() | C_out["Price"].isna()].head(20))

# 5) 导出
from pathlib import Path
out_dir = base_dir / "output"
out_dir.mkdir(exist_ok=True)

c_path = out_dir / "C_final.xlsx"
with pd.ExcelWriter(c_path, engine="openpyxl") as w:
    C_out.to_excel(w, index=False, sheet_name="C")

print("Saved:", c_path)

Rows: 101
Missing Qty: 31
Missing Price: 31
Sample rows not filled (first 20):


,Product,Suppl category,Description,Size,Pack,Store,Price,Qty
1,2221.0,18221,MAGGI Thick Soy Sauce Nuoc Tuong Dau美極越南醬油,700ML,12.0,0/6,NaN,<NA>
4,6827.0,77711,WANG Korean BBQ Sauce for Beef Short Rib,480g,15.0,0/0,NaN,<NA>
5,6836.0,60391,34514WANG Stick Rice Cake Tteokboki Tteock王牌韩国年糕条,600g,12.0,0/0,NaN,<NA>
10,9368.0,64481,RICO Bubble Milk Tea Drink Honeydew Can,350g,24.0,0/0,NaN,<NA>
12,9406.0,11211,TYJ Happy Belly Gyoza Skin饺子皮 100mm,300g,30.0,0/13,NaN,<NA>
15,114116.0,14691,LID for 2L/4L ice cream Tub IC4L两-四升雪糕盒盖子,1PCS,1.0,-1/0,NaN,<NA>
20,149321.0,17511,17511MENG FU Pork Floss-Original 蒙福原味肉松,90g,40.0,0/0,NaN,<NA>
21,149327.0,65621,65621MENG FU Taiwanese Pork Sausage 163Origina...,430g,16.0,0/0,NaN,<NA>
22,149328.0,65661,65661Meng Fu Chicken Sausage 蒙福火锅脆皮亲亲肠,360g,16.0,0/0,NaN,<NA>
25,220009.0,45311,Chaokoh Coconut Milk Powder查哥泰国奶丝,370g,12.0,0/0,NaN,<NA>


Saved: /Users/xuyali/Documents/个人资料/Work/自媒体/Total Asia/output/C_final.xlsx


In [26]:
import pandas as pd
from pathlib import Path
import numpy as np

use_matched = matched_all if "matched_all" in globals() else matched

# ---- 1) 计算 Product 目标长度（强制 int）----
# 用字符串长度计算，避免出现 float
prod_len = int(df_b["product"].astype(str).str.replace(r"\.0$", "", regex=True).str.len().max())

def zfill_prod(x, width=prod_len):
    if pd.isna(x):
        return None
    s = str(x).strip()

    # 处理像 401.0 这种
    s = s.replace(",", "")
    if s.endswith(".0"):
        s = s[:-2]

    # 如果出现科学计数法（少见，但保险）
    if "e" in s.lower():
        try:
            s = str(int(float(s)))
        except Exception:
            pass

    return s.zfill(int(width))

# ---- 2) 统一 B 的 product_key（保留前导0）----
df_b_keyed = df_b.copy()
df_b_keyed["product_key"] = df_b_keyed["product"].apply(zfill_prod)

# ---- 3) 统一 matched 的 B_Product key ----
map_df = use_matched[["B_Product", "A_QTY", "A_PRICE"]].copy()
map_df = map_df.dropna(subset=["B_Product"])
map_df["product_key"] = map_df["B_Product"].apply(zfill_prod)

# 若重复命中：保留第一条
map_df = map_df.drop_duplicates(subset=["product_key"], keep="first")

qty_map = dict(zip(map_df["product_key"], map_df["A_QTY"]))
price_map = dict(zip(map_df["product_key"], map_df["A_PRICE"]))

# ---- 4) 生成 C（保持 B 顺序）并回填 A 的 Qty/Price ----
C = df_b_keyed.copy()
C["Qty"] = C["product_key"].map(qty_map)
C["Price"] = C["product_key"].map(price_map)

# ---- 5) 仅输出指定列，并插入索引列（第一列）----
C_out = pd.DataFrame({
    "Product": C["product_key"],
    "Suppl category": C["suppl_category"],
    "Description": C["desc_b"],
    "Size": C["size_b"],
    "Pack": C.get("pack_b", np.nan),
    "Store": C.get("store", np.nan),
    "Price": C["Price"],  # A 的
    "Qty": C["Qty"],      # A 的
})

C_out.insert(0, "Index", range(1, len(C_out) + 1))

# ---- Checkpoint ----
print("prod_len:", prod_len)
print("Sample Product:", C_out["Product"].head(10).tolist())
print("Missing Qty:", C_out["Qty"].isna().sum(), "Missing Price:", C_out["Price"].isna().sum())

# ---- 6) 导出 ----
out_dir = base_dir / "output"
out_dir.mkdir(exist_ok=True)
c_path = out_dir / "C_final.xlsx"

with pd.ExcelWriter(c_path, engine="openpyxl") as w:
    C_out.to_excel(w, index=False, sheet_name="C")

print("Saved:", c_path)

prod_len: 7
Sample Product: ['0000401', '0002221', '0002933', '0003413', '0006827', '0006836', '0007603', '0007675', '0008831', '0009214']
Missing Qty: 31 Missing Price: 31
Saved: /Users/xuyali/Documents/个人资料/Work/自媒体/Total Asia/output/C_final.xlsx
